# 01 – Exploration de l'API Fiscal Data (Allan)

Objectif : comprendre ce que renvoie l'API des adjudications avant d'écrire le nettoyage.

1. Télécharger **une** page et regarder sa structure (`meta`, `data`).
2. Comprendre les ~120 colonnes : types, valeurs manquantes, colonnes récentes.
3. Télécharger **toutes** les pages avec `telecharger_brut()` et regarder les types de titres.

In [12]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook lancé depuis notebooks/
from donnees_adjudications import URL_API, telecharger_ou_lire, telecharger_brut

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 1. Une page de l'API

On demande les 100 adjudications les plus récentes (tri décroissant sur la date).

In [13]:
params = {"format": "json", "sort": "-auction_date", "page[size]": 100, "page[number]": 1}
reponse = telecharger_ou_lire("exploration_page.json", params=params)

print("Clés de la réponse :", list(reponse.keys()))
print("Clés de meta       :", list(reponse["meta"].keys()))

Clés de la réponse : ['data', 'meta', 'links']
Clés de meta       : ['count', 'labels', 'dataTypes', 'dataFormats', 'total-count', 'total-pages']


In [14]:
meta = reponse["meta"]
print("Lignes sur cette page :", meta["count"])
print("Lignes au total       :", meta["total-count"])
print("Nombre de pages       :", meta["total-pages"], "(à 100 lignes par page)")

Lignes sur cette page : 100
Lignes au total       : 11154
Nombre de pages       : 112 (à 100 lignes par page)


`data` est une liste de dictionnaires, un par adjudication. Voici le premier :

In [15]:
reponse["data"][0]

{'record_date': '2026-10-30',
 'cusip': '91282CRR4',
 'security_type': 'Note',
 'security_term': '5-Year',
 'auction_date': '2026-10-22',
 'issue_date': '2026-10-30',
 'maturity_date': '2031-10-15',
 'price_per100': 'null',
 'accrued_int_per100': 'null',
 'accrued_int_per1000': 'null',
 'adj_accrued_int_per1000': 'null',
 'adj_price': 'null',
 'allocation_pctage': 'null',
 'allocation_pctage_decimals': '2',
 'announcemtd_cusip': 'null',
 'announcemt_date': '2026-10-15',
 'auction_format': 'Single-Price',
 'avg_med_discnt_rate': 'null',
 'avg_med_investment_rate': 'null',
 'avg_med_price': 'null',
 'avg_med_discnt_margin': 'null',
 'avg_med_yield': 'null',
 'back_dated': 'Yes',
 'back_dated_date': '2026-10-15',
 'bid_to_cover_ratio': 'null',
 'callable': 'No',
 'call_date': 'null',
 'called_date': 'null',
 'cash_management_bill_cmb': 'No',
 'closing_time_comp': '01:00 PM',
 'closing_time_noncomp': '12:00 PM',
 'comp_accepted': 'null',
 'comp_bid_decimals': '3',
 'comp_tendered': 'null',

## 2. Les colonnes

In [16]:
df = pd.DataFrame(reponse["data"])
print(df.shape)
df.head(3)

(100, 114)


,record_date,cusip,security_type,security_term,auction_date,issue_date,maturity_date,price_per100,accrued_int_per100,accrued_int_per1000,adj_accrued_int_per1000,adj_price,allocation_pctage,allocation_pctage_decimals,announcemtd_cusip,announcemt_date,auction_format,avg_med_discnt_rate,avg_med_investment_rate,avg_med_price,avg_med_discnt_margin,avg_med_yield,back_dated,back_dated_date,bid_to_cover_ratio,callable,call_date,called_date,cash_management_bill_cmb,closing_time_comp,closing_time_noncomp,comp_accepted,comp_bid_decimals,comp_tendered,comp_tenders_accepted,corpus_cusip,cpi_base_reference_period,currently_outstanding,dated_date,direct_bidder_accepted,direct_bidder_tendered,est_pub_held_mat_by_type_amt,fima_included,fima_noncomp_accepted,fima_noncomp_tendered,first_int_period,first_int_payment_date,floating_rate,frn_index_determination_date,frn_index_determination_rate,high_discnt_rate,high_investment_rate,high_price,high_discnt_margin,high_yield,index_ratio_on_issue_date,indirect_bidder_accepted,indirect_bidder_tendered,int_payment_frequency,int_rate,low_discnt_rate,low_investment_rate,low_price,low_discnt_margin,low_yield,mat_date,max_comp_award,max_noncomp_award,max_single_bid,min_bid_amt,min_strip_amt,min_to_issue,multiples_to_bid,multiples_to_issue,nlp_exclusion_amt,nlp_reporting_threshold,noncomp_accepted,noncomp_tenders_accepted,offering_amt,original_cusip,original_dated_date,original_issue_date,original_security_term,pdf_filenm_announcemt,pdf_filenm_comp_results,pdf_filenm_noncomp_results,primary_dealer_accepted,primary_dealer_tendered,ref_cpi_on_dated_date,ref_cpi_on_issue_date,reopening,security_term_day_month,security_term_week_year,series,soma_accepted,soma_holdings,soma_included,soma_tendered,spread,std_int_payment_per1000,strippable,tiin_conversion_factor_per1000,total_accepted,total_tendered,treas_retail_accepted,treas_retail_tenders_accepted,unadj_accrued_int_per1000,unadj_price,xml_filenm_announcemt,xml_filenm_comp_results,inflation_index_security,tint_cusip_1,tint_cusip_2,pdf_filenm_spec_announcemt
0,2026-10-30,91282CRR4,Note,5-Year,2026-10-22,2026-10-30,2031-10-15,null,null,null,null,null,null,2,null,2026-10-15,Single-Price,null,null,null,null,null,Yes,2026-10-15,null,No,null,null,No,01:00 PM,12:00 PM,null,3,null,Yes,912821VJ1,1982-1984=100,null,2026-10-15,null,null,0,Yes,null,null,Normal,2027-04-15,No,null,null,null,null,null,null,null,1.001540,null,null,Semi-Annual,null,null,null,null,null,null,2026-10-30,9100000000,10000000,9100000000,100,100,100,100,100,0,9100000000,null,Yes,26000000000,null,null,null,5-Year,A_20261015_1.pdf,null,NCR_20261022_3.pdf,null,null,334.397610,334.911480,No,0-Month,5-Year,AE-2031,null,0,No,null,null,null,Yes,null,null,null,null,Yes,null,null,A_20261015_1.xml,null,Yes,null,null,null
1,2026-10-23,912810UX4,Bond,19-Year 10-Month,2026-10-21,2026-10-23,2046-08-15,null,null,9.6093700000,null,null,null,2,null,2026-10-15,Single-Price,null,null,null,null,null,Yes,2026-08-15,null,No,null,null,No,01:00 PM,12:00 PM,null,3,null,Yes,912803HY2,null,31053000000.000000,2026-08-15,null,null,0,Yes,null,null,Normal,2027-02-15,No,null,null,null,null,null,null,null,null,null,null,Semi-Annual,5.125000,null,null,null,null,null,2026-10-23,4550000000,10000000,4550000000,100,100,100,100,100,10200000000,4550000000,null,Yes,13000000000,null,2026-08-15,2026-08-31,20-Year,A_20261015_3.pdf,null,NCR_20261021_2.pdf,null,null,null,null,Yes,10-Month,19-Year,Bonds of August 2046,null,0,No,null,null,25.6250,Yes,null,null,null,null,Yes,null,null,A_20261015_3.xml,null,No,null,null,null
2,2026-10-22,912797VA2,Bill,6-Week,2026-10-20,2026-10-22,2026-12-03,null,null,null,null,null,null,2,null,2026-10-15,Single-Price,null,null,null,null,null,null,null,null,null,null,null,No,11:30 AM,11:00 AM,null,3,null,Yes,null,null,179421000000.000000,null,null,null,241706000000,Yes,null,null,null,null,No,null,null,null,null,null,null,null,null,null,null,None,null,null,null,null,null,null,2026-10-22,33250000000,10000000,33250000000,10

L'API renvoie **tout en texte**, nombres compris. `meta["dataTypes"]` indique le type prévu pour chaque colonne, `meta["labels"]` son libellé.

In [17]:
colonnes = pd.DataFrame({
    "libelle": pd.Series(meta["labels"]),
    "type_prevu": pd.Series(meta["dataTypes"]),
    "type_recu": df.dtypes.astype(str),
})
colonnes

,libelle,type_prevu,type_recu
record_date,Record Date,DATE,str
cusip,CUSIP,STRING,str
security_type,Security Type,STRING,str
security_term,Security Term,STRING,str
auction_date,Auction Date,DATE,str
issue_date,Issue Date,DATE,str
maturity_date,Maturity Date,DATE,str
price_per100,Price per $100,STRING,str
accrued_int_per100,Accrued Interest per $100,NUMBER,str
accrued_int_per1000,"Accrued Interest per $1,000",NUMBER,str


Valeurs manquantes : l'API écrit la chaîne `"null"`, pas une vraie valeur manquante. On compte donc les `"null"` par colonne.

In [18]:
nb_null = (df == "null").sum().sort_values(ascending=False)
print(f"{(nb_null == len(df)).sum()} colonnes entièrement 'null' sur ces 100 lignes récentes")
nb_null[nb_null > 0]

9 colonnes entièrement 'null' sur ces 100 lignes récentes


announcemtd_cusip                 100
avg_med_investment_rate           100
call_date                         100
called_date                       100
avg_med_price                     100
low_price                         100
tint_cusip_2                      100
original_cusip                    100
low_investment_rate               100
accrued_int_per100                 98
adj_accrued_int_per1000            98
unadj_price                        98
tiin_conversion_factor_per1000     98
unadj_accrued_int_per1000          98
adj_price                          98
cpi_base_reference_period          97
frn_index_determination_rate       97
avg_med_discnt_margin              97
high_discnt_margin                 97
frn_index_determination_date       97
index_ratio_on_issue_date          97
low_discnt_margin                  97
ref_cpi_on_dated_date              97
ref_cpi_on_issue_date              97
spread                             97
pdf_filenm_spec_announcemt         97
tint_cusip_1

## 3. Toutes les pages

In [19]:
brut = telecharger_brut()
print(brut.shape)
print("Types reçus :", brut.dtypes.astype(str).unique())

(11154, 114)
Types reçus : <ArrowStringArray>
['str']
Length: 1, dtype: str


In [20]:
print("Première adjudication :", brut["auction_date"].min())
print("Dernière adjudication  :", brut["auction_date"].max())
brut["security_type"].value_counts()

Première adjudication : 1979-10-31
Dernière adjudication  : 2026-10-22


security_type
Bill    8361
Note    2338
Bond     455
Name: count, dtype: int64

Titres par type et par terme : seules les notes et les bonds nominaux sont comparables entre eux (étape 5, dans `nettoyer()`).

In [21]:
pd.crosstab(brut["security_type"], brut["security_term"]).T.head(40)

security_type,Bill,Bond,Note
security_term,,,
1-Day,7,0,0
1-Year 10-Month,0,0,52
1-Year 11-Month,0,0,52
10-Day,9,0,0
10-Year,0,0,223
101-Day,1,0,0
102-Day,1,0,0
103-Day,4,0,0
105-Day,42,0,0


Part de `"null"` par colonne selon la décennie : permet de voir quelles colonnes n'existent que depuis 2000 ou 2009.

In [22]:
decennie = brut["auction_date"].str[:3] + "0s"
part_null = (brut == "null").groupby(decennie).mean().T
part_null[(part_null > 0).any(axis=1)].round(2)

auction_date,1970s,1980s,1990s,2000s,2010s,2020s
price_per100,1.00,1.00,0.79,0.00,0.00,0.00
accrued_int_per100,1.00,1.00,1.00,1.00,0.98,0.98
accrued_int_per1000,0.50,0.77,0.79,0.86,0.88,0.90
adj_accrued_int_per1000,1.00,1.00,0.99,0.97,0.96,0.97
adj_price,1.00,1.00,0.99,0.97,0.96,0.97
allocation_pctage,0.00,0.00,0.00,0.00,0.00,0.00
allocation_pctage_decimals,0.00,0.00,0.00,0.00,0.00,0.00
announcemtd_cusip,1.00,1.00,1.00,1.00,0.99,1.00
avg_med_discnt_rate,0.50,0.23,0.22,0.19,0.33,0.25
avg_med_investment_rate,0.50,0.23,0.31,1.00,1.00,1.00
